In [2]:
import pandas as pd
import numpy as np
from scipy import stats

In [3]:
file_name = "Данные для тестового задания.xlsx"

excel_file = pd.ExcelFile(file_name)
excel_file.sheet_names

['Данные об аудитории', 'Данные АБ тестов', 'Листеры']

In [4]:
audience = pd.read_excel(
    file_name,
    sheet_name="Данные об аудитории"
)

audience.head()

,date,user_id,view_adverts
0,2023-11-11,8c020470-8461-11ed-83d0-552e8cc749d6,13
1,2023-11-18,5875f070-7b92-11ee-a6fb-8b298e83f4f7,14
2,2023-11-29,3c2d27c0-4fd6-11eb-b89f-2ffb31b67dd6,21
3,2023-11-29,234a96d0-ad16-11ed-a2e6-793ddfeeba1f,23
4,2023-11-29,4d07c180-644f-11eb-879c-b7c02edf4f37,12


In [5]:
audience.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16814 entries, 0 to 16813
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          16814 non-null  datetime64[ns]
 1   user_id       16814 non-null  object        
 2   view_adverts  16814 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 394.2+ KB


. Во вкладке "Данные об аудитории" информация о пользователях, посетивших наше приложение в ноябре. Чему равен MAU продукта? 
*MAU (Monthly Active Users) — это метрика, используемая для измерения активности пользователей в течение одного месяца. Она показывает количество уникальных пользователей, которые взаимодействовали с продуктом, сервисом или приложением хотя бы один раз за последний месяц.


In [6]:
mau = audience["user_id"].nunique()
print("MAU:", mau)

MAU: 7639


2. Используя вкладку "Данные об аудитории", посчитайте, чему будет равен DAU 
*DAU (Daily Active Users) — это метрика, которая показывает количество уникальных пользователей, которые взаимодействовали с продуктом, приложением или сервисом хотя бы один раз в течение дня. DAU помогает понять, сколько пользователей активно пользуются продуктом каждый день.
 255 490 560 483


In [7]:
dau_by_day = audience.groupby("date")["user_id"].nunique()

dau_by_day

date
2023-11-01    623
2023-11-02    649
2023-11-03    573
2023-11-04    343
2023-11-05    350
2023-11-06    660
2023-11-07    629
2023-11-08    600
2023-11-09    661
2023-11-10    583
2023-11-11    354
2023-11-12    377
2023-11-13    646
2023-11-14    687
2023-11-15    690
2023-11-16    639
2023-11-17    585
2023-11-18    412
2023-11-19    378
2023-11-20    711
2023-11-21    651
2023-11-22    608
2023-11-23    632
2023-11-24    595
2023-11-25    366
2023-11-26    372
2023-11-27    589
2023-11-28    610
2023-11-29    621
2023-11-30    620
Name: user_id, dtype: int64

In [8]:
average_dau = dau_by_day.mean()

print("Средний DAU:", average_dau)
print("Средний DAU после округления:", round(average_dau))

Средний DAU: 560.4666666666667
Средний DAU после округления: 560


3. Используя вкладку "Данные об аудитории", посчитайте, чему будет равен retention первого дня у пользователей, пришедших в продукт 1 ноября 
*Retention (удержание пользователей) — это метрика, которая показывает, сколько пользователей продолжает пользоваться продуктом через определенный промежуток времени после первоначального взаимодействия. Retention можно рассчитать как процент пользователей, вернувшихся в продукт через определенное время (например, через 1 день, 1 неделю, 1 месяц) от количества всех новых пользователей.
 28,3% 26,6% 38,5% 32,7%


In [9]:
cohort_date = pd.Timestamp("2023-11-01")
day_1_date = cohort_date + pd.Timedelta(days=1)

# Определяем дату первого посещения каждого пользователя
first_visit = audience.groupby("user_id")["date"].min()

# Пользователи, впервые пришедшие 1 ноября
cohort_users = set(
    first_visit[first_visit == cohort_date].index
)

# Пользователи, активные 2 ноября
day_1_users = set(
    audience.loc[
        audience["date"] == day_1_date,
        "user_id"
    ]
)

# Пользователи из когорты 1 ноября, вернувшиеся 2 ноября
retained_users = cohort_users.intersection(day_1_users)

retention_day_1 = len(retained_users) / len(cohort_users) * 100

print("Размер когорты 1 ноября:", len(cohort_users))
print("Вернулись 2 ноября:", len(retained_users))
print("Retention первого дня:", round(retention_day_1, 1), "%")

Размер когорты 1 ноября: 623
Вернулись 2 ноября: 166
Retention первого дня: 26.6 %


5. Во вкладке "Данные об аудитории" есть информация о том, сколько объявлений посмотрел каждый пользователь (view_adverts). Посчитайте пользовательскую конверсию в просмотр объявления за ноябрь? (в пользователях) 
* Пользовательская конверсия — это метрика, которая показывает, какой процент пользователей выполнил целевое действие по отношению к общему количеству пользователей. В контексте веб-сайтов это может быть действие, такое как просмотр объявления или клик по рекламному баннеру. 
 41,8% 54,7% 46,3% 39%


In [10]:
# Суммарное количество просмотров каждого пользователя за ноябрь
views_by_user = audience.groupby("user_id")["view_adverts"].sum()

# Количество пользователей, посмотревших хотя бы одно объявление
converted_users = (views_by_user > 0).sum()

# Общее количество уникальных пользователей
total_users = views_by_user.size

# Пользовательская конверсия
conversion = converted_users / total_users * 100

print("Всего пользователей:", total_users)
print("Посмотрели хотя бы одно объявление:", converted_users)
print("Конверсия:", round(conversion, 1), "%")

Всего пользователей: 7639
Посмотрели хотя бы одно объявление: 3538
Конверсия: 46.3 %


6. Используя информацию из вкладки "Данные об аудитории", посчитайте среднее количество просмотренных объявлений на пользователя в ноябре
 4,9 6,2 5,3 2,9


In [11]:
# Суммарные просмотры каждого пользователя за ноябрь
views_by_user = audience.groupby("user_id")["view_adverts"].sum()

# Среднее количество просмотров на одного пользователя
average_views = views_by_user.mean()

print("Среднее количество просмотров:", round(average_views, 1))

Среднее количество просмотров: 2.9


In [12]:
average_views_check = (
    audience["view_adverts"].sum()
    / audience["user_id"].nunique()
)

print(round(average_views_check, 1))

2.9


7. Мы провели опрос среди 2000 пользователей. Из них 500 «критики», 1200 «сторонники» и 300 «нейтралы». Посчитайте, чему будет равен NPS 
* NPS (Net Promoter Score) — это метрика, которая измеряет лояльность пользователей к компании или продукту и делит их на три группы: Сторонники (Promoters) , Нейтралы (Passives),  Критики (Detractors). NPS высчитывается как (% сторонников - % критиков).
 30% 43% 40% 35%


In [13]:
total = 2000
promoters = 1200
detractors = 500

nps = promoters / total * 100 - detractors / total * 100

print("NPS:", round(nps), "%")

NPS: 35 %


In [ ]:
8. Во вкладке "Данные АБ-тестов" результаты трех несвязанных АБ тестов для ARPU (общая выручка/общее количество пользователей).
Посмотрите на результаты тестов и интерпретируйте их. Напишите значения p-value, которые вы получили.
Подготовьте выводы и рекомендации. 

experiment_num - номер эксперимента
experiment_group - группа, в которую попал пользователь
user_id - id пользователя
revenue - выручка, которую сгенерировал пользователь, купив платную услугу продвижения


In [14]:
ab_tests = pd.read_excel(
    file_name,
    sheet_name="Данные АБ тестов"
)

ab_tests.head()

,experiment_num,experiment_group,user_id,revenue
0,1,test,38456,520
1,1,control,13125924,806
2,1,control,9761984,0
3,1,test,11387012,208
4,1,test,18319648,104


In [15]:
ab_tests.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2835 entries, 0 to 2834
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   experiment_num    2835 non-null   int64 
 1   experiment_group  2835 non-null   object
 2   user_id           2835 non-null   int64 
 3   revenue           2835 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 88.7+ KB


In [16]:
print("Номера экспериментов:", ab_tests["experiment_num"].unique())
print("Группы:", ab_tests["experiment_group"].unique())

ab_tests.groupby(
    ["experiment_num", "experiment_group"]
).agg(
    users=("user_id", "nunique"),
    rows=("user_id", "size"),
    total_revenue=("revenue", "sum"),
    arpu=("revenue", "mean")
)

Номера экспериментов: [1 2 3]
Группы: ['test' 'control']


users  rows  total_revenue        arpu
experiment_num experiment_group                                        
1              control             465   465         335944  722.460215
               test                480   480         319555  665.739583
2              control             465   465         327664  704.653763
               test                480   480         159806  332.929167
3              control             465   465         308391  663.206452
               test                480   480         479361  998.668750

In [17]:
for experiment in [1, 2, 3]:
    control = ab_tests.loc[
        (ab_tests["experiment_num"] == experiment) &
        (ab_tests["experiment_group"] == "control"),
        "revenue"
    ]

    test = ab_tests.loc[
        (ab_tests["experiment_num"] == experiment) &
        (ab_tests["experiment_group"] == "test"),
        "revenue"
    ]

    t_stat, p_value = stats.ttest_ind(
        test,
        control,
        equal_var=False
    )

    uplift = (test.mean() / control.mean() - 1) * 100

    print(f"Эксперимент {experiment}")
    print(f"ARPU control: {control.mean():.2f}")
    print(f"ARPU test: {test.mean():.2f}")
    print(f"Изменение ARPU: {uplift:.2f}%")
    print(f"p-value: {p_value:.6f}")
    print("-" * 30)

Эксперимент 1
ARPU control: 722.46
ARPU test: 665.74
Изменение ARPU: -7.85%
p-value: 0.688966
------------------------------
Эксперимент 2
ARPU control: 704.65
ARPU test: 332.93
Изменение ARPU: -52.75%
p-value: 0.001128
------------------------------
Эксперимент 3
ARPU control: 663.21
ARPU test: 998.67
Изменение ARPU: 50.58%
p-value: 0.060315
------------------------------


9. По датасету с листерами посчитайте средний доход на пользователя 
 121.2 156.4 70.9 30.7 средняя здесь не применима



In [18]:
listers = pd.read_excel(
    file_name,
    sheet_name="Листеры"
)

listers.head()

,user_id,date,cnt_adverts,age,cnt_contacts,revenue
0,100,2022-01-01,6,21,119,53
1,100,2022-01-02,2,21,200,18
2,100,2022-01-03,6,21,193,42
3,100,2022-01-04,2,21,143,38
4,100,2022-01-05,2,21,190,40


In [19]:
listers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   user_id       158 non-null    int64         
 1   date          158 non-null    datetime64[ns]
 2   cnt_adverts   158 non-null    int64         
 3   age           158 non-null    int64         
 4   cnt_contacts  158 non-null    int64         
 5   revenue       158 non-null    int64         
dtypes: datetime64[ns](1), int64(5)
memory usage: 7.5 KB


In [20]:
listers.describe(include="all")

,user_id,date,cnt_adverts,age,cnt_contacts,revenue
count,158.000000,158,158.000000,158.000000,158.000000,158.000000
mean,115.101266,2022-01-14 12:45:34.177215232,2.886076,27.424051,94.031646,30.702532
min,100.000000,2022-01-01 00:00:00,0.000000,20.000000,0.000000,0.000000
25%,107.000000,2022-01-07 00:00:00,1.000000,22.750000,42.000000,17.000000
50%,115.000000,2022-01-14 00:00:00,3.000000,28.000000,92.500000,31.000000
75%,123.000000,2022-01-21 18:00:00,5.000000,31.000000,148.500000,45.000000
max,130.000000,2022-01-30 00:00:00,6.000000,36.000000,200.000000,60.000000
std,9.008976,NaN,2.028380,4.640303,60.231128,16.575060


In [21]:
revenue_by_user = listers.groupby("user_id")["revenue"].sum()

average_revenue_per_user = revenue_by_user.mean()

print("Уникальных пользователей:", listers["user_id"].nunique())
print("Общий доход:", listers["revenue"].sum())
print(
    "Средний доход на пользователя:",
    round(average_revenue_per_user, 2)
)

Уникальных пользователей: 31
Общий доход: 4851
Средний доход на пользователя: 156.48


In [ ]:
10. По датасету с листерами посчитайте медиану возраста пользователя 
 27,42 28 27,93 27 медиана здесь не применима


In [22]:
# Оставляем по одной записи на каждого пользователя
age_by_user = (
    listers[["user_id", "age"]]
    .drop_duplicates(subset="user_id")
)

median_age = age_by_user["age"].median()

print("Количество пользователей:", len(age_by_user))
print("Медиана возраста:", median_age)

Количество пользователей: 31
Медиана возраста: 28.0


18. Были получены следующие результаты. Коллеги просят вас подтвердить их и сделать окончательный вывод по эксперименту.
●	Вариант A (контрольная группа) — 100 047 501 посетитель, 1003 платежа.
●	Вариант B (тестовая группа) — 100 001 055 посетителей, 1099 платежей.
Какие рекомендации вы бы дали, основываясь на этих данных?
Ваш ответ:



In [23]:
visitors_a = 100_047_501
payments_a = 1003

visitors_b = 100_001_055
payments_b = 1099

conversion_a = payments_a / visitors_a
conversion_b = payments_b / visitors_b

# Относительное изменение конверсии
uplift = (conversion_b / conversion_a - 1) * 100

# Объединённая конверсия
pooled_conversion = (
    (payments_a + payments_b)
    / (visitors_a + visitors_b)
)

# Стандартная ошибка
standard_error = np.sqrt(
    pooled_conversion
    * (1 - pooled_conversion)
    * (1 / visitors_a + 1 / visitors_b)
)

# z-статистика и двусторонний p-value
z_stat = (
    conversion_b - conversion_a
) / standard_error

p_value = 2 * stats.norm.sf(abs(z_stat))

print(f"Конверсия A: {conversion_a:.6%}")
print(f"Конверсия B: {conversion_b:.6%}")
print(f"Относительный рост: {uplift:.2f}%")
print(f"z-статистика: {z_stat:.4f}")
print(f"p-value: {p_value:.6f}")

Конверсия A: 0.001003%
Конверсия B: 0.001099%
Относительный рост: 9.62%
z-статистика: 2.1046
p-value: 0.035330
